# 加载模块

In [1]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
import re
import os
import json
import random

# 不息屏
import pyautogui
import threading

# 防休眠

## 防休眠函数

In [2]:
def keep_awake():
    """每2分钟轻微移动鼠标防止休眠"""
    print("🖱️ 启动防休眠模式...")
    while True:
        try:
            # 轻微移动鼠标位置（不会影响使用）
            pyautogui.moveRel(1, 0)
            pyautogui.moveRel(-1, 0)
            print(f"⏰ 防休眠心跳: {time.strftime('%H:%M:%S')}")
            time.sleep(120)  # 2分钟移动一次
        except Exception as e:
            print(f"防休眠出错: {e}")
            break


## 执行防休眠

In [3]:
# 启动防休眠线程
awake_thread = threading.Thread(target=keep_awake, daemon=True)
awake_thread.start()
print("✅ 防休眠已启动，电脑不会息屏")

🖱️ 启动防休眠模式...
✅ 防休眠已启动，电脑不会息屏
⏰ 防休眠心跳: 01:13:21


# 设置Chrome浏览器驱动

In [4]:
def setup_driver():
    options = webdriver.ChromeOptions()
    # 可选：无头模式（不显示浏览器界面）
    # options.add_argument('--headless')
    options.add_argument('--disable-blink-features=AutomationControlled')
    options.add_argument('--disable-extensions')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36')
    
    try:
        driver = webdriver.Chrome(options=options)
        driver.maximize_window()
        return driver
    except Exception as e:
        print(f"驱动设置失败: {e}")
        raise

# 初始化浏览器
driver = setup_driver()
print("浏览器初始化完成")


浏览器初始化完成


# 获取所有板块URL

## 爬取函数

In [5]:
# 单元格2：修复版的获取所有区和板块URL
def get_all_districts_and_areas(base_url):
    """获取所有区和板块的URL"""
    print("正在获取所有区和板块信息...")
    driver.get(base_url)
    time.sleep(random.uniform(2, 4))
    
    all_districts = []
    
    try:
        # 等待区域选择元素加载
        wait = WebDriverWait(driver, 10)
        district_container = wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "div.position > dl:nth-child(2) > dd > div"))
        )
        
        # 获取所有区的链接（排除第一个"不限"）
        area_links = district_container.find_elements(By.CSS_SELECTOR, "div:nth-child(1) > a")
        # 过滤掉"不限"
        area_links = [link for link in area_links if link.text.strip() and link.text.strip() != "不限"]
        print(f"找到 {len(area_links)} 个区: {[link.text.strip() for link in area_links]}")
        
        # 遍历每个区
        for area_idx in range(len(area_links)):
            # 每次重新获取元素，避免stale element reference
            driver.get(base_url)
            time.sleep(random.uniform(1, 3))
            
            # 重新等待区域选择元素加载
            district_container = WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "div.position > dl:nth-child(2) > dd > div"))
            )
            
            # 重新获取区链接
            area_links = district_container.find_elements(By.CSS_SELECTOR, "div:nth-child(1) > a")
            area_links = [link for link in area_links if link.text.strip() and link.text.strip() != "不限"]
            
            if area_idx >= len(area_links):
                print(f"区索引 {area_idx} 超出范围")
                continue
                
            current_area_link = area_links[area_idx]
            area_name = current_area_link.text.strip()
            
            print(f"\n正在处理区: {area_name} ({area_idx+1}/{len(area_links)})")
            
            try:
                # 获取区URL
                area_href = current_area_link.get_attribute("href")
                if not area_href.startswith('http'):
                    area_href = "https://gz.lianjia.com" + area_href
                
                print(f"点击区: {area_name}")
                # 使用JavaScript点击，更稳定
                driver.execute_script("arguments[0].click();", current_area_link)
                time.sleep(random.uniform(1.5, 3.5))  # 等待页面加载
                
                # 等待板块区域加载完成
                WebDriverWait(driver, 10).until(
                    EC.presence_of_element_located((By.CSS_SELECTOR, "div.position > dl:nth-child(2) > dd > div > div:nth-child(2)"))
                )
                
                # 获取当前区的所有板块
                district_links = driver.find_elements(By.CSS_SELECTOR, "div.position > dl:nth-child(2) > dd > div > div:nth-child(2) > a")
                # 过滤掉"不限"
                district_links = [link for link in district_links if link.text.strip() and link.text.strip() != "不限"]
                
                print(f"在区 {area_name} 中找到 {len(district_links)} 个板块")
                
                for district_link in district_links:
                    district_name = district_link.text.strip()
                    district_url = district_link.get_attribute("href")
                    if not district_url.startswith('http'):
                        district_url = "https://gz.lianjia.com" + district_url
                    
                    all_districts.append({
                        'area': area_name,
                        'name': district_name,
                        'url': district_url,
                        'area_url': area_href
                    })
                    print(f"  找到板块: {district_name}")
                
            except Exception as e:
                print(f"处理区 {area_name} 时出错: {e}")
                continue
                
    except Exception as e:
        print(f"获取区和板块信息时出错: {e}")
        import traceback
        traceback.print_exc()
    
    return all_districts

## 执行爬取

In [6]:
# # 执行获取区和板块
# base_url = "https://gz.lianjia.com/chengjiao/changxing1/"
# all_districts = get_all_districts_and_areas(base_url)

# # 统计结果
# if all_districts:
#     areas = set(district['area'] for district in all_districts)
#     print(f"\n统计结果:")
#     print(f"共找到 {len(areas)} 个区")
#     print(f"共找到 {len(all_districts)} 个板块")
    
#     # 按区统计板块数量
#     from collections import defaultdict
#     area_stats = defaultdict(list)
#     for district in all_districts:
#         area_stats[district['area']].append(district['name'])
    
#     for area, districts in area_stats.items():
#         print(f"  {area}: {len(districts)} 个板块")
        
#     # 显示前几个板块作为示例
#     print(f"\n前10个板块示例:")
#     for i, district in enumerate(all_districts[:10]):
#         print(f"  {i+1}. {district['area']} - {district['name']} - {district['url']}")
# else:
#     print("没有找到任何板块信息")

In [7]:
# pd.DataFrame(all_districts)[['area', 'name', 'url', 'area_url']].to_excel("广州链家板块列表.xlsx", index=False)

# 房屋信息解析函数

In [5]:
def parse_house_info(house_element):
    """解析单个房屋的详细信息"""
    house_data = {}
    
    try:
        # 标题信息（包含小区名字、户型、面积）
        title_element = house_element.find_element(By.CSS_SELECTOR, "div.title > a")
        title_text = title_element.text.strip()
        house_data['title'] = title_text
        
        # 解析小区名字、户型、面积
        if ' ' in title_text:
            parts = title_text.split(' ')
            house_data['community'] = parts[0] if len(parts) > 0 else ""
            house_data['layout'] = parts[1] if len(parts) > 1 else ""
            house_data['area'] = parts[2] if len(parts) > 2 else ""
        else:
            house_data['community'] = title_text
            house_data['layout'] = ""
            house_data['area'] = ""
        
        # 房屋详细信息（朝向、装修等）
        try:
            house_info_element = house_element.find_element(By.CSS_SELECTOR, "div.houseInfo")
            house_info_text = house_info_element.text.strip()
            house_data['house_info'] = house_info_text
            
            # 解析详细信息
            info_parts = house_info_text.split(' | ')
            if len(info_parts) >= 3:
                house_data['orientation'] = info_parts[1] if len(info_parts) > 1 else ""
                house_data['decoration'] = info_parts[2] if len(info_parts) > 2 else ""
            else:
                house_data['orientation'] = ""
                house_data['decoration'] = ""
                
        except NoSuchElementException:
            house_data['house_info'] = ""
            house_data['orientation'] = ""
            house_data['decoration'] = ""
        
        # 成交日期
        try:
            deal_date_element = house_element.find_element(By.CSS_SELECTOR, "div.dealDate")
            house_data['deal_date'] = deal_date_element.text.strip()
        except NoSuchElementException:
            house_data['deal_date'] = ""
        
        # 成交总价
        try:
            total_price_element = house_element.find_element(By.CSS_SELECTOR, "div.totalPrice > span")
            house_data['total_price'] = total_price_element.text.strip()
        except NoSuchElementException:
            house_data['total_price'] = ""
        
        # 位置信息
        try:
            position_element = house_element.find_element(By.CSS_SELECTOR, "div.positionInfo")
            house_data['position'] = position_element.text.strip()
        except NoSuchElementException:
            house_data['position'] = ""
        
        # 成交单价
        try:
            unit_price_element = house_element.find_element(By.CSS_SELECTOR, "div.unitPrice")
            house_data['unit_price'] = unit_price_element.text.strip()
        except NoSuchElementException:
            house_data['unit_price'] = ""
        
        # 楼层和建成年份
        try:
            flood_element = house_element.find_element(By.CSS_SELECTOR, "div.flood")
            flood_text = flood_element.text.strip()
            house_data['floor_info'] = flood_text
            
            # 尝试提取建成年份
            year_match = re.search(r'(\d{4})年', flood_text)
            house_data['built_year'] = year_match.group(1) if year_match else ""
        except NoSuchElementException:
            house_data['floor_info'] = ""
            house_data['built_year'] = ""
        
        # 交易信息（满五年、近地铁等）
        try:
            deal_house_info = house_element.find_element(By.CSS_SELECTOR, "div.dealHouseInfo")
            deal_text = deal_house_info.text.strip()
            house_data['deal_conditions'] = deal_text
            house_data['five_years'] = "满五年" in deal_text
            house_data['near_subway'] = "近地铁" in deal_text
        except NoSuchElementException:
            house_data['deal_conditions'] = ""
            house_data['five_years'] = False
            house_data['near_subway'] = False
        
        # 挂牌价和成交周期
        try:
            deal_cycle_info = house_element.find_element(By.CSS_SELECTOR, "div.dealCycleeInfo")
            cycle_text = deal_cycle_info.text.strip()
            house_data['deal_cycle_info'] = cycle_text
            
            # 解析挂牌价和成交周期
            list_price_match = re.search(r'挂牌(\d+)万', cycle_text)
            cycle_match = re.search(r'成交周期(\d+)天', cycle_text)
            
            house_data['list_price'] = list_price_match.group(1) if list_price_match else ""
            house_data['deal_cycle_days'] = cycle_match.group(1) if cycle_match else ""
        except NoSuchElementException:
            house_data['deal_cycle_info'] = ""
            house_data['list_price'] = ""
            house_data['deal_cycle_days'] = ""
        
        # 中介信息
        try:
            agent_element = house_element.find_element(By.CSS_SELECTOR, "div.agentInfoList > a")
            house_data['agent'] = agent_element.text.strip()
        except NoSuchElementException:
            house_data['agent'] = ""
            
    except Exception as e:
        print(f"解析房屋信息时出错: {e}")
        return None
    
    return house_data


# 断点续爬功能

In [6]:
def save_checkpoint(district_index, page_index, houses_data, filename="checkpoint.json"):
    """保存爬取进度"""
    checkpoint = {
        'district_index': district_index,
        'page_index': page_index,
        'total_houses': len(houses_data),
        'save_time': time.strftime("%Y-%m-%d %H:%M:%S")
    }
    
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(checkpoint, f, ensure_ascii=False, indent=2)
    
    # 同时保存数据到Excel
    if houses_data:
        pd.DataFrame(houses_data).to_excel("临时成交数据.xlsx", index=False)
        
    print(f"检查点已保存: 板块 {district_index}, 页面 {page_index}, 总数据 {len(houses_data)} 条")

def load_checkpoint(filename="checkpoint.json"):
    """加载爬取进度"""
    if os.path.exists(filename):
        try:
            with open(filename, 'r', encoding='utf-8') as f:
                checkpoint = json.load(f)
            print(f"找到检查点: 板块 {checkpoint['district_index']}, 页面 {checkpoint['page_index']}")
            return checkpoint
        except Exception as e:
            print(f"加载检查点失败: {e}")
    return None

def save_partial_data(houses_data, filename="partial_data.json"):
    """保存部分爬取的数据"""
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(houses_data, f, ensure_ascii=False, indent=2)
    print(f"部分数据已保存: {len(houses_data)} 条记录")

def load_partial_data(filename="partial_data.json"):
    """加载部分爬取的数据"""
    if os.path.exists(filename):
        try:
            with open(filename, 'r', encoding='utf-8') as f:
                data = json.load(f)
            print(f"加载部分数据: {len(data)} 条记录")
            return data
        except Exception as e:
            print(f"加载部分数据失败: {e}")
    return []

def save_final_data(houses_data, filename="链家成交数据.xlsx"):
    """保存最终数据到Excel"""
    if not houses_data:
        print("没有数据可保存")
        return
    
    df = pd.DataFrame(houses_data)
    
    # 重新排列列的顺序
    columns_order = [
        'district', 'community', 'layout', 'area', 'orientation', 'decoration',
        'total_price', 'unit_price', 'list_price', 'deal_date', 'deal_cycle_days',
        'floor_info', 'built_year', 'position', 'five_years', 'near_subway',
        'agent', 'title', 'house_info', 'deal_conditions', 'deal_cycle_info',
        'district_url'
    ]
    
    existing_columns = [col for col in columns_order if col in df.columns]
    df = df[existing_columns]
    
    df.to_excel(filename, index=False, engine='openpyxl')
    print(f"数据已保存到 {filename}，共 {len(houses_data)} 条记录")


# 主爬取函数

## 主爬函数

In [7]:
def is_captcha_page(driver):
    """检测是否为人机验证页面"""
    page_source = driver.page_source
    # 使用你提供的具体关键词
    captcha_indicators = ["人机验证", "本次访问已触发人机验证，请按指示操作"]
    return any(indicator in page_source for indicator in captcha_indicators)

def wait_for_captcha_solve(driver, check_interval=10):
    """等待用户手动完成人机验证"""
    while is_captcha_page(driver):
        print(f"⏳ 仍在人机验证页面，{check_interval}秒后重新检查...")
        time.sleep(check_interval)
    print("✅ 人机验证已通过，继续爬取...")

In [13]:
# # 单元格5：主爬取函数
# def crawl_with_checkpoint(all_districts, start_district=0, start_page=1, max_pages_per_district=100):
#     """带断点续爬功能的主爬取函数"""
    
#     global driver
    
#     # 加载已有的数据
#     all_houses = load_partial_data()
    
#     # 加载检查点
#     checkpoint = load_checkpoint()
#     if checkpoint:
#         current_district = checkpoint['district_index']
#         current_page = checkpoint['page_index']
#         print(f"从断点继续: 板块 {current_district}, 页面 {current_page}")
#     else:
#         current_district = start_district
#         current_page = start_page
#         print(f"开始新的爬取: 板块 {current_district}, 页面 {current_page}")
    
#     # 从断点开始爬取
#     for district_idx in range(current_district, len(all_districts)):
#         district = all_districts[district_idx]
#         district_houses = []
        
#         print(f"\n[{district_idx+1}/{len(all_districts)}] 正在处理板块: {district['name']}")
        
#         # 确定起始页面
#         start_page_num = current_page if district_idx == current_district else 1
        
#         try:
#             for page in range(start_page_num, max_pages_per_district + 1):
#                 page_url = f"{district['url']}pg{page}/" if page > 1 else district['url']
#                 print(f"正在爬取第 {page} 页: {page_url}")
                
#                 # driver.get(page_url)
#                 # time.sleep(random.uniform(1.5, 3.5))

#                 try:
#                     driver.get(page_url)
#                     if is_captcha_page(driver):
#                         print("⚠️ 遇到人机验证，等待手动处理...")
#                         wait_for_captcha_solve(driver)

#                     time.sleep(random.uniform(1.5, 3.5))
#                 except Exception as e:
#                     print(f"❌ 浏览器会话失效，重新初始化...")
#                     # 重新初始化浏览器
#                     driver.quit()
#                     time.sleep(random.uniform(1.5, 3.5))
#                     driver = setup_driver()
#                     driver.get(page_url)
#                     if is_captcha_page(driver):
#                         print("⚠️ 遇到人机验证，等待手动处理...")
#                         wait_for_captcha_solve(driver)
#                     time.sleep(random.uniform(1.5, 4.5))
                
#                 # # 检查是否被反爬
#                 # if "验证" in driver.title or "验证" in driver.page_source:
#                 #     print("遇到反爬验证，暂停程序")
#                 #     save_checkpoint(district_idx, page, all_houses)
#                 #     save_partial_data(all_houses)
#                 #     return all_houses

#                 # # 反爬检测
#                 # page_text = driver.page_source
#                 # if any(keyword in page_text for keyword in ["验证码", "人机验证", "antispider", "请输入验证码"]):
#                 #     print("⚠️ 可能遇到反爬验证，暂停程序")
#                 #     save_checkpoint(district_idx, page, all_houses)
#                 #     save_partial_data(all_houses)
#                 #     return all_houses
                
#                 # 检查页面是否有数据
#                 if "没有找到房源" in driver.page_source:
#                     print("该板块没有找到房源数据")
#                     break
                
#                 # 等待房屋列表加载
#                 wait = WebDriverWait(driver, 10)
#                 house_list = wait.until(
#                     EC.presence_of_element_located((By.CSS_SELECTOR, "ul.listContent"))
#                 )
                
#                 # 获取所有房屋元素
#                 house_elements = house_list.find_elements(By.TAG_NAME, "li")
#                 print(f"第 {page} 页找到 {len(house_elements)} 个房屋")
                
#                 page_houses = []
#                 for i, house_element in enumerate(house_elements):
#                     try:
#                         house_data = parse_house_info(house_element)
#                         if house_data:
#                             house_data['district'] = district['name']
#                             house_data['district_url'] = district['url']
#                             page_houses.append(house_data)
#                             print(f"  已解析第 {i+1} 个房屋: {house_data.get('community', '')}")
#                     except Exception as e:
#                         print(f"解析第 {i+1} 个房屋时出错: {e}")
                
#                 district_houses.extend(page_houses)
                
#                 # 检查是否有下一页
#                 # try:
#                 #     next_page = driver.find_element(By.CSS_SELECTOR, "a.next")
#                 #     if not next_page.is_enabled():
#                 #         print("没有更多页面，转到下一个板块")
#                 #         break
#                 # except NoSuchElementException:
#                 #     print("没有找到下一页按钮，转到下一个板块")
#                 #     break

#                 # 检查是否有下一页
#                 if page < max_pages_per_district:
#                     # 尝试访问下一页看看是否存在
#                     next_page_url = f"{district['url']}pg{page+1}/"
#                     print(f"尝试检查下一页: {next_page_url}")
                    
#                     # 保存当前URL
#                     # current_url = driver.current_url
                    
#                     try:
#                         driver.get(next_page_url)
#                          # 检查是否遇到人机验证
#                         if is_captcha_page(driver):
#                             print("⚠️ 遇到人机验证，等待手动处理...")
#                             wait_for_captcha_solve(driver)
#                         time.sleep(random.uniform(1.5, 2.5))
                        
#                         # 检查下一页是否有数据
#                         house_list_check = driver.find_elements(By.CSS_SELECTOR, "ul.listContent li")
#                         if house_list_check:
#                             print(f"第 {page+1} 页有数据，继续爬取")
#                             # 回到当前页继续正常流程
#                             driver.get(current_url)
#                             time.sleep(random.uniform(1, 3.5))
#                         else:
#                             print("没有更多页面，转到下一个板块")
#                             break
#                     except:
#                         print("无法访问下一页，转到下一个板块")
#                         break
#                 else:
#                     print("达到最大页数限制，转到下一个板块")
#                     break
               
                
#                 # 保存检查点
#                 save_checkpoint(district_idx, page + 1, all_houses + district_houses)
                
#                 time.sleep(random.uniform(1.5, 3.5))  # 页面间延迟
            
#             # 当前板块完成，重置页面计数器
#             current_page = 1
            
#             # 保存当前板块的数据
#             all_houses.extend(district_houses)
#             save_partial_data(all_houses)
#             print(f"板块 {district['name']} 完成，共获取 {len(district_houses)} 条数据")
            
#             time.sleep(random.uniform(1.5, 3.5))  # 板块间延迟
            
#         except Exception as e:
#             print(f"爬取板块 {district['name']} 时出错: {e}")
#             save_checkpoint(district_idx, page, all_houses)
#             save_partial_data(all_houses)
#             return all_houses
    
#     # 所有板块完成，清理检查点文件
#     if os.path.exists("checkpoint.json"):
#         os.remove("checkpoint.json")
#         print("检查点文件已清理")
    
#     return all_houses

In [ ]:
def parse_current_page_data(district, district_houses, house_elements):
    """解析当前页面的房屋数据"""
    page_houses = []
    for i, house_element in enumerate(house_elements):
        try:
            house_data = parse_house_info(house_element)
            if house_data:
                house_data['district'] = district['name']
                house_data['district_url'] = district['url']
                page_houses.append(house_data)
                print(f"  已解析第 {i+1} 个房屋: {house_data.get('community', '')}")
        except Exception as e:
            print(f"解析第 {i+1} 个房屋时出错: {e}")
    
    district_houses.extend(page_houses)

In [ ]:
# 单元格5：主爬取函数
def crawl_with_checkpoint(all_districts, start_district=0, start_page=1, max_pages_per_district=100):
    """带断点续爬功能的主爬取函数"""
    
    global driver
    
    # 加载已有的数据
    all_houses = load_partial_data()
    
    # 加载检查点
    checkpoint = load_checkpoint()
    if checkpoint:
        current_district = checkpoint['district_index']
        current_page = checkpoint['page_index']
        print(f"从断点继续: 板块 {current_district}, 页面 {current_page}")
    else:
        current_district = start_district
        current_page = start_page
        print(f"开始新的爬取: 板块 {current_district}, 页面 {current_page}")
    
    # 从断点开始爬取
    for district_idx in range(current_district, len(all_districts)):
        district = all_districts[district_idx]
        district_houses = []
        
        print(f"\n[{district_idx+1}/{len(all_districts)}] 正在处理板块: {district['name']}")
        
        # 确定起始页面
        start_page_num = current_page if district_idx == current_district else 1
        
        try:
            for page in range(start_page_num, max_pages_per_district + 1):
                page_url = f"{district['url']}pg{page}/" if page > 1 else district['url']
                print(f"正在爬取第 {page} 页: {page_url}")
                
                # driver.get(page_url)
                # time.sleep(random.uniform(1.5, 3.5))

                try:
                    driver.get(page_url)
                    if is_captcha_page(driver):
                        print("⚠️ 遇到人机验证，等待手动处理...")
                        wait_for_captcha_solve(driver)

                    time.sleep(random.uniform(1.5, 3.5))
                except Exception as e:
                    print(f"❌ 浏览器会话失效，重新初始化...")
                    # 重新初始化浏览器
                    driver.quit()
                    time.sleep(random.uniform(1.5, 3.5))
                    driver = setup_driver()
                    driver.get(page_url)
                    if is_captcha_page(driver):
                        print("⚠️ 遇到人机验证，等待手动处理...")
                        wait_for_captcha_solve(driver)
                    time.sleep(random.uniform(1.5, 4.5))
                
                # # 检查是否被反爬
                # if "验证" in driver.title or "验证" in driver.page_source:
                #     print("遇到反爬验证，暂停程序")
                #     save_checkpoint(district_idx, page, all_houses)
                #     save_partial_data(all_houses)
                #     return all_houses

                # # 反爬检测
                # page_text = driver.page_source
                # if any(keyword in page_text for keyword in ["验证码", "人机验证", "antispider", "请输入验证码"]):
                #     print("⚠️ 可能遇到反爬验证，暂停程序")
                #     save_checkpoint(district_idx, page, all_houses)
                #     save_partial_data(all_houses)
                #     return all_houses
                
                # 检查页面是否有数据
                if "没有找到房源" in driver.page_source:
                    print("该板块没有找到房源数据")
                    break
                
                # 等待房屋列表加载
                wait = WebDriverWait(driver, 10)
                house_list = wait.until(
                    EC.presence_of_element_located((By.CSS_SELECTOR, "ul.listContent"))
                )
                
                # 获取所有房屋元素
                house_elements = house_list.find_elements(By.TAG_NAME, "li")
                print(f"第 {page} 页找到 {len(house_elements)} 个房屋")
                
                # 检查是否有下一页
                if page < max_pages_per_district:
                    # 尝试访问下一页看看是否存在
                    next_page_url = f"{district['url']}pg{page+1}/"
                    print(f"尝试访问下一页: {next_page_url}")
                    
                    try:
                        driver.get(next_page_url)
                        time.sleep(random.uniform(1.5, 2.5))
                        
                        # 检查是否遇到人机验证
                        if is_captcha_page(driver):
                            print("⚠️ 遇到人机验证，等待手动处理...")
                            wait_for_captcha_solve(driver)
                        
                        # 检查下一页是否有数据
                        if "没有找到房源" in driver.page_source:
                            print("没有更多页面，转到下一个板块")
                            # 在退出前解析当前页的数据
                            parse_current_page_data(district, district_houses, house_elements)
                            break
                            
                        # 等待房屋列表加载
                        wait = WebDriverWait(driver, 10)
                        house_list = wait.until(
                            EC.presence_of_element_located((By.CSS_SELECTOR, "ul.listContent"))
                        )
                        
                        house_list_check = house_list.find_elements(By.TAG_NAME, "li")
                        if house_list_check:
                            print(f"第 {page+1} 页有数据，直接在该页面解析")
                            # 先解析当前页的数据
                            parse_current_page_data(district, district_houses, house_elements)
                            # 然后在下一页解析数据
                            page_houses = []
                            for i, house_element in enumerate(house_list_check):
                                try:
                                    house_data = parse_house_info(house_element)
                                    if house_data:
                                        house_data['district'] = district['name']
                                        house_data['district_url'] = district['url']
                                        page_houses.append(house_data)
                                        print(f"  已解析第 {i+1} 个房屋: {house_data.get('community', '')}")
                                except Exception as e:
                                    print(f"解析第 {i+1} 个房屋时出错: {e}")
                            
                            district_houses.extend(page_houses)
                            # 更新page变量，因为我们现在已经在下一页了
                            page += 1
                        else:
                            print("没有更多页面，转到下一个板块")
                            # 在退出前解析当前页的数据
                            parse_current_page_data(district, district_houses, house_elements)
                            break
                    except:
                        print("无法访问下一页，转到下一个板块")
                        # 在退出前解析当前页的数据
                        parse_current_page_data(district, district_houses, house_elements)
                        break
                else:
                    print("达到最大页数限制，转到下一个板块")
                    # 在退出前解析当前页的数据
                    parse_current_page_data(district, district_houses, house_elements)
                    break
               
                
                # 保存检查点
                save_checkpoint(district_idx, page + 1, all_houses + district_houses)
                
                time.sleep(random.uniform(1.5, 3.5))  # 页面间延迟
            
            # 当前板块完成，重置页面计数器
            current_page = 1
            
            # 保存当前板块的数据
            all_houses.extend(district_houses)
            save_partial_data(all_houses)
            print(f"板块 {district['name']} 完成，共获取 {len(district_houses)} 条数据")
            
            time.sleep(random.uniform(1.5, 3.5))  # 板块间延迟
            
        except Exception as e:
            print(f"爬取板块 {district['name']} 时出错: {e}")
            save_checkpoint(district_idx, page, all_houses)
            save_partial_data(all_houses)
            return all_houses
    
    # 所有板块完成，清理检查点文件
    if os.path.exists("checkpoint.json"):
        os.remove("checkpoint.json")
        print("检查点文件已清理")
    
    return all_houses

## 执行主爬

In [ ]:
# 单元格6：执行爬取
print("开始链家成交数据爬取...")

df = pd.read_excel("广州链家板块列表.xlsx")
all_districts = df.to_dict('records')

# 执行爬取（可以随时中断，重新运行这个单元格会从中断处继续）
final_data = crawl_with_checkpoint(all_districts, max_pages_per_district=100)

# 保存最终结果
if final_data:
    save_final_data(final_data, "广州链家成交数据.xlsx")
    print(f"\n爬取完成！共获取 {len(final_data)} 条成交数据")
else:
    print("没有获取到数据")

开始链家成交数据爬取...
加载部分数据: 70450 条记录
找到检查点: 板块 150, 页面 18
从断点继续: 板块 150, 页面 18

[151/254] 正在处理板块: 沙湾
正在爬取第 18 页: https://gz.lianjia.com/chengjiao/shawan1/pg18/
⚠️ 遇到人机验证，等待手动处理...
⏳ 仍在人机验证页面，10秒后重新检查...
✅ 人机验证已通过，继续爬取...
第 18 页找到 30 个房屋
  已解析第 1 个房屋: 香缤豪庭
  已解析第 2 个房屋: 祈福水城
  已解析第 3 个房屋: 祈福水城
  已解析第 4 个房屋: 敏建锦绣世家
  已解析第 5 个房屋: 中华大道翠园西
  已解析第 6 个房屋: 雅居乐锦官城雍容南台
  已解析第 7 个房屋: 金沙丽水
  已解析第 8 个房屋: 荷景花园丽景居
  已解析第 9 个房屋: 敏建锦绣世家
  已解析第 10 个房屋: 敏建锦绣世家
  已解析第 11 个房屋: 祈福水城
  已解析第 12 个房屋: 敏建锦绣世家
  已解析第 13 个房屋: 金沙诚筑
  已解析第 14 个房屋: 紫坭糖厂
  已解析第 15 个房屋: 荷景花园二区
  已解析第 16 个房屋: 金沙湾花园
  已解析第 17 个房屋: 荷景花园二区
  已解析第 18 个房屋: 源福轩
  已解析第 19 个房屋: 金沙湾花园
  已解析第 20 个房屋: 紫坭糖厂
  已解析第 21 个房屋: 荷景花园一区
  已解析第 22 个房屋: 金沙湾花园
  已解析第 23 个房屋: 祈福水城
  已解析第 24 个房屋: 荷景花园六区
  已解析第 25 个房屋: 荷景花园一区
  已解析第 26 个房屋: 雅居乐锦官城雍容南台
  已解析第 27 个房屋: 敏建锦绣世家
  已解析第 28 个房屋: 祈福水城
  已解析第 29 个房屋: 雅居乐锦官城雍容南台
  已解析第 30 个房屋: 荷景花园一区
尝试访问下一页: https://gz.lianjia.com/chengjiao/shawan1/pg19/
第 19 页有数据，直接在该页面解析
  已解析第 1 个房屋: 源福轩
  已解析第 2 个房屋: 翠怡轩
  已解析第 3 个房屋: 雅居乐锦

In [12]:
# 单元格7：关闭浏览器（最后执行）
driver.quit()
print("浏览器已关闭")

浏览器已关闭


# 账号

## API云短信平台

https://www.api21k.com/user/sms/apiBuy

这个好用：https://www.eomsg.com/appweb/t1.html